In [ ]:
import deeplabcut 
import pandas as pd
import numpy as np

from pathlib import Path

phase = 3
deeplabcut_project = Path('../data/dlc/mouse_face-telfer-2023-06-27').resolve()
assert deeplabcut_project.exists()

In [ ]:
deeplabcut.analyze_time_lapse_frames(
    deeplabcut_project / 'config.yaml',
    deeplabcut_project / f'labeled-data/phase{phase}')

In [ ]:
reference_labels = pd.read_hdf(deeplabcut_project / 'labeled-data/phase1/CollectedData_telfer.h5')
reference_labels.head(3)

In [ ]:
dlc_file = next(deeplabcut_project.glob(f'labeled-data/phase{phase}/*DLC*.h5'))
generated_labels = pd.read_hdf(dlc_file)
generated_labels.head(3)


In [ ]:
images = generated_labels.index

index = pd.MultiIndex.from_tuples(('labeled-data', f'phase{phase}', l) for l in images)
columns = pd.MultiIndex.from_tuples(
    [('telfer',b,c) for s,b,c in generated_labels.columns],
    names=['scorer', 'bodyparts', 'coords']
)

reshaped_df = generated_labels.copy()
reshaped_df = reshaped_df.set_index(index)
reshaped_df.columns = columns
reshaped_df.head(3)

### Identify when left-right confusion occurs and filter lower confidence points

In [ ]:
counter = 0
for idx, row in reshaped_df.iterrows():
    pairs = [
        ['left_ear', 'right_ear'],
        ['left_eye', 'right_eye']
    ]

    threshold = 5
    for bp1, bp2 in pairs:
        x1, y1, c1 = row.loc[:, bp1, ['x', 'y', 'likelihood']]
        x2, y2, c2 = row.loc[:, bp2, ['x', 'y', 'likelihood']]
        distance = np.sqrt((x2-x1)**2 + (y2-y1)**2)
        
        if distance is not np.nan and distance < 5:
            if c1 > c2:
                reshaped_df.loc[idx, pd.IndexSlice[:,bp2,:]] = np.nan
            elif c2 > c1:
                reshaped_df.loc[idx, pd.IndexSlice[:,bp1,:]] = np.nan
            
            counter += 1
            
        
print(counter)
            


In [ ]:
likelihood = reshaped_df.xs('likelihood', level='coords', axis=1)
confident_labels = reshaped_df.where(likelihood > 0.6)
images = confident_labels.index
confident_labels = confident_labels.drop(columns='likelihood', level='coords')
confident_labels

In [ ]:
confident_labels.to_hdf(deeplabcut_project / f'labeled-data/phase{phase}/CollectedData_telfer.h5', '/keypoints')
confident_labels.to_csv(deeplabcut_project / f'labeled-data/phase{phase}/CollectedData_telfer.csv')

## Apply Trained Model to Videos Test

In [ ]:
videos_dir = Path("../data/raw/")
videos = list(map(str, videos_dir.glob('*.mp4')))

deeplabcut.analyze_videos(
    deeplabcut_project / 'config.yaml',
    videos,
    destfolder=Path("../data/raw/")
)